# 04 — Real Data Inference

Examine inference results on the 7 real streams:
1. Per-stream posterior samples and corner plots
2. Log evidence matrix
3. Model comparison (Bayes factors)
4. Inter-stream consistency

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'outputs'

STREAMS = ['GD1', 'Pal5', 'Orphan', 'ATLAS', 'Jhelum', 'Fjorm', 'Sylgr']
DM_MODELS = ['CDM', 'WDM', 'FDM', 'SIDM']

## Log evidence heatmap

In [ ]:
import seaborn as sns

log_ev = pd.read_csv(OUT / 'log_evidences.csv', index_col=0)
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(log_ev, annot=True, fmt='.2f', cmap='RdYlGn',
            center=log_ev.values.mean(), ax=ax, linewidths=0.5)
ax.set_title('Log Evidence: Stream x DM Model')
plt.tight_layout()
plt.show()

# Best model per stream
best = log_ev.idxmax(axis=1)
for s, m in best.items():
    print(f'{s:8s} -> {m} (log Z = {log_ev.loc[s, m]:.3f})')

## GD-1 posteriors under each DM model

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
colors = {'CDM': '#1f77b4', 'WDM': '#ff7f0e', 'FDM': '#2ca02c', 'SIDM': '#d62728'}

for ax, model in zip(axes.flat, DM_MODELS):
    samples_path = OUT / 'GD1' / f'samples_{model}.csv'
    if samples_path.exists():
        samples = pd.read_csv(samples_path)
        for col in samples.columns:
            ax.hist(samples[col], bins=50, alpha=0.6, label=col, density=True)
    ax.set_title(f'GD-1 posterior: {model}')
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

plt.suptitle('GD-1 Posterior Distributions', y=1.02)
plt.tight_layout()
plt.show()

## Credible intervals comparison across streams

In [ ]:
# Load all credible intervals for CDM
model = 'CDM'
rows = []
for stream in STREAMS:
    ci_path = OUT / stream / f'credible_intervals_{model}.csv'
    if ci_path.exists():
        ci = pd.read_csv(ci_path, index_col=0)
        for param in ci.index:
            rows.append({
                'stream': stream, 'param': param,
                'median': ci.loc[param, 'median'],
                'lower': ci.loc[param, 'lower'],
                'upper': ci.loc[param, 'upper'],
            })

ci_df = pd.DataFrame(rows)

# Plot for the first physics parameter
params = ci_df['param'].unique()
fig, axes = plt.subplots(1, len(params), figsize=(6*len(params), 5))
if len(params) == 1:
    axes = [axes]

for ax, param in zip(axes, params):
    sub = ci_df[ci_df['param'] == param]
    y = range(len(sub))
    ax.errorbar(sub['median'], y,
                xerr=[sub['median'] - sub['lower'], sub['upper'] - sub['median']],
                fmt='o', capsize=4, color=colors.get(model, 'blue'))
    ax.set_yticks(list(y))
    ax.set_yticklabels(sub['stream'])
    ax.set_xlabel(param)
    ax.set_title(f'{model}: {param} per stream')

plt.tight_layout()
plt.show()

## n_impacts consistency across streams

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, model in zip(axes, DM_MODELS):
    for stream in STREAMS:
        sp = OUT / stream / f'samples_{model}.csv'
        if sp.exists():
            s = pd.read_csv(sp)
            if 'n_impacts' in s.columns:
                ax.hist(s['n_impacts'], bins=np.arange(-0.5, 15.5, 1),
                        alpha=0.4, density=True, label=stream)
    ax.set_title(f'{model}')
    ax.set_xlabel('n_impacts')
    ax.set_ylabel('Density')
    ax.legend(fontsize=6, ncol=2)

plt.suptitle('n_impacts posterior across streams', y=1.02)
plt.tight_layout()
plt.show()

## Model posterior probabilities

In [ ]:
mc = pd.read_csv(OUT / 'model_comparison.csv')
display(mc.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
probs = mc.set_index('model')['posterior_probability']
ax.bar(probs.index, probs.values,
       color=[colors[m] for m in probs.index])
ax.set_ylabel('Posterior probability')
ax.set_title('DM Model Posterior Probabilities')
for i, (m, p) in enumerate(probs.items()):
    ax.text(i, p + 0.01, f'{p:.1%}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()